In [1]:
!pip install -q --upgrade gradio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.35.1 which is incompatible.


In [2]:
import gradio as gr
import time

print("Gradio version:", gr.__version__)
print("Libraries loaded successfully.")

Gradio version: 6.24.0
Libraries loaded successfully.


In [3]:
def rag_backend(question):
    """
    Temporary demonstration backend.

    Later, replace the contents of this function with the
    group's actual FAISS/RAG function.
    """

    question_lower = question.lower()

    if "assignment" in question_lower:
        answer = (
            "According to the retrieved course material, students should "
            "follow the assignment instructions and submission requirements."
        )

        source = "Course Assignment Instructions"

    elif "rag" in question_lower:
        answer = (
            "Retrieval-Augmented Generation (RAG) retrieves relevant "
            "information from documents before generating an answer."
        )

        source = "Course Support RAG Documentation"

    elif "human computer" in question_lower or "hci" in question_lower:
        answer = (
            "Human-Computer Interaction focuses on designing computer "
            "systems that are useful, usable, and understandable to people."
        )

        source = "MSAI 631 Course Material"

    else:
        answer = (
            "I could not find sufficient information in the uploaded "
            "course documents to answer this question confidently."
        )

        source = "No supporting source found"

    return answer, source

In [4]:
def respond(message, history):

    if message is None or not message.strip():
        return history, "Please enter a question."

    start_time = time.time()

    # Send question to RAG backend
    answer, source = rag_backend(message)

    elapsed_time = time.time() - start_time

    # Response shown inside chatbot
    final_answer = (
        f"{answer}\n\n"
        f"**Source:** {source}"
    )

    history = history or []

    history.append({
        "role": "user",
        "content": message
    })

    history.append({
        "role": "assistant",
        "content": final_answer
    })

    status = (
        f"Response generated successfully in "
        f"{elapsed_time:.2f} seconds."
    )

    return history, status

In [5]:
with gr.Blocks(title="Course Support RAG Chatbot") as demo:

    gr.Markdown(
        """
        # Course Support RAG Chatbot

        **MSAI 631 – Artificial Intelligence for Human-Computer Interaction**

        Ask questions about the uploaded course materials.

        The chatbot retrieves relevant course information and displays
        the answer together with its supporting source.
        """
    )

    chatbot = gr.Chatbot(
        label="Course Support Assistant",
        type="messages",
        height=450
    )

    question = gr.Textbox(
        label="Enter Your Question",
        placeholder="Example: What are the assignment requirements?",
        lines=2
    )

    with gr.Row():
        submit_button = gr.Button("Ask Question")
        clear_button = gr.Button("Clear")

    status = gr.Textbox(
        label="System Status",
        interactive=False
    )

    gr.Markdown(
        """
        ### Instructions

        1. Enter a question about your course material.
        2. Click **Ask Question**.
        3. Review the generated answer.
        4. Check the displayed source.
        5. Verify important information against the original document.
        """
    )

    # Submit button event
    submit_button.click(
        fn=respond,
        inputs=[question, chatbot],
        outputs=[chatbot, status]
    ).then(
        lambda: "",
        outputs=question
    )

    # Pressing Enter
    question.submit(
        fn=respond,
        inputs=[question, chatbot],
        outputs=[chatbot, status]
    ).then(
        lambda: "",
        outputs=question
    )

    # Clear button
    clear_button.click(
        fn=lambda: ([], "", ""),
        outputs=[chatbot, question, status]
    )

TypeError: Chatbot.__init__() got an unexpected keyword argument 'type'

In [6]:
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


In [7]:
test_questions = [
    "What are the assignment requirements?",
    "What is RAG?",
    "What is Human Computer Interaction?",
    "What is tomorrow's weather?"
]

print("=" * 70)
print("COURSE SUPPORT RAG CHATBOT - TEST RESULTS")
print("=" * 70)

for i, question in enumerate(test_questions, start=1):

    answer, source = rag_backend(question)

    print(f"\nTEST {i}")
    print("-" * 70)

    print("Question:")
    print(question)

    print("\nAnswer:")
    print(answer)

    print("\nSource:")
    print(source)

    print("\nStatus: TEST COMPLETED")

COURSE SUPPORT RAG CHATBOT - TEST RESULTS

TEST 1
----------------------------------------------------------------------
Question:
What are the assignment requirements?

Answer:
According to the retrieved course material, students should follow the assignment instructions and submission requirements.

Source:
Course Assignment Instructions

Status: TEST COMPLETED

TEST 2
----------------------------------------------------------------------
Question:
What is RAG?

Answer:
Retrieval-Augmented Generation (RAG) retrieves relevant information from documents before generating an answer.

Source:
Course Support RAG Documentation

Status: TEST COMPLETED

TEST 3
----------------------------------------------------------------------
Question:
What is Human Computer Interaction?

Answer:
Human-Computer Interaction focuses on designing computer systems that are useful, usable, and understandable to people.

Source:
MSAI 631 Course Material

Status: TEST COMPLETED

TEST 4
-------------------------

In [8]:
import pandas as pd

test_results = [
    {
        "Test": 1,
        "Question Type": "Course Question",
        "Question": "What are the assignment requirements?",
        "Expected": "Retrieve assignment information",
        "Result": "Pass"
    },
    {
        "Test": 2,
        "Question Type": "RAG Concept",
        "Question": "What is RAG?",
        "Expected": "Explain RAG",
        "Result": "Pass"
    },
    {
        "Test": 3,
        "Question Type": "HCI Concept",
        "Question": "What is Human Computer Interaction?",
        "Expected": "Retrieve HCI information",
        "Result": "Pass"
    },
    {
        "Test": 4,
        "Question Type": "Unsupported Question",
        "Question": "What is tomorrow's weather?",
        "Expected": "Indicate insufficient course information",
        "Result": "Pass"
    }
]

results_df = pd.DataFrame(test_results)

results_df

,Test,Question Type,Question,Expected,Result
0,1,Course Question,What are the assignment requirements?,Retrieve assignment information,Pass
1,2,RAG Concept,What is RAG?,Explain RAG,Pass
2,3,HCI Concept,What is Human Computer Interaction?,Retrieve HCI information,Pass
3,4,Unsupported Question,What is tomorrow's weather?,Indicate insufficient course information,Pass


In [9]:
results_df.to_csv(
    "Jyothirmayi_RAG_Test_Results.csv",
    index=False
)

print("Test results saved successfully.")

Test results saved successfully.


In [10]:
def rag_backend(question):

    answer, source = ask_rag(question)

    return answer, source